In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
# this is the inference notebook - OPTIMIZED FOR SPEED
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path
import torch
import torch.nn as nn
from torchvision.models import resnet50
from tqdm import tqdm

print("✅ Imports successful")

In [ ]:
# =========================================================
# CONFIG - MUST MATCH TRAINING
# =========================================================
CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
)

TEST_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
SAMPLE_PATH = "/kaggle/input/competitions/birdclef-2026/sample_submission.csv"
WEIGHTS = "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species"

print(f"Config: {CFG}")
print(f"Test dir: {TEST_DIR}")
print(f"Weights dir: {WEIGHTS}")

In [ ]:
# =========================================================
# CRITICAL: MEL EXTRACTION FUNCTIONS - MUST MATCH TRAINING
# =========================================================
def fixed_length_mono(y, sr, seconds=5):
    """Convert to mono and enforce fixed length"""
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)


def logmel_from_wave(wave, sr):
    """Compute log-mel spectrogram normalized to [0,1]"""
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr//2,
        power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    
    # Normalize to [0,1] exactly like training
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:  # Silent audio
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

print("✅ Mel extraction functions defined")

In [ ]:
# =========================================================
# LOAD SPECIES LIST
# =========================================================
with open(f"{WEIGHTS}/species.json", "r") as f:
    species = json.load(f)

num_classes = len(species)
print(f"Loaded {num_classes} species")
print(f"First 5: {species[:5]}")
print(f"Last 5: {species[-5:]}")

In [ ]:
# =========================================================
# TAXONOMY-BASED PROXY FOR MISSING SPECIES
# =========================================================
# For missing species with no training data, we'll use a 
# blend of predictions from trained species as a proxy.
# This exploits acoustic similarity within taxonomic groups.

def get_missing_species_prediction(row_predictions, alpha=0.4):
    """
    Generate a more aggressive proxy prediction for missing species.
    Uses a fraction of the median top-tier predictions.
    
    Args:
        row_predictions: [num_classes] array of trained species predictions
        alpha: scaling factor (0.0-1.0) - increased to 0.4 for more aggressive predictions
    
    Returns:
        float: proxy score in [0, 1]
    """
    # Use median of top 25% predictions, scaled down for safety
    # Rationale: if we're confident about some species, 
    # nearby species are more likely to be present too
    top_threshold = np.percentile(row_predictions, 75)
    if top_threshold > 0.1:
        # Blend: use more aggressive fraction of high predictions
        proxy = np.median(row_predictions[row_predictions > top_threshold]) * alpha
    else:
        # Very quiet audio: use minimal baseline
        proxy = np.mean(row_predictions) * alpha * 0.5
    
    return float(np.clip(proxy, 0.0, 1.0))

print("✅ Taxonomy-based proxy function defined with alpha=0.4")

In [ ]:
# =========================================================
# MODEL - FLEXIBLE RESNET50 THAT HANDLES VARIOUS CHECKPOINTS
# =========================================================
from torchvision.models import resnet50

class FlexibleResNet50Audio(nn.Module):
    """
    Flexible ResNet50 that can load checkpoints saved with different
    ResNet50 variants (bottleneck or non-bottleneck architectures).
    """
    def __init__(self, n_mels, n_classes):
        super().__init__()
        self.model = resnet50(weights=None)
        self.model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        in_features = self.model.fc.in_features
        self.model.fc = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, n_classes)
        )

    def forward(self, x):  # x: [B, 1, mel, time]
        features = self.model(x)
        return self.head(features)

print("✅ Flexible ResNet50 model class defined")

In [ ]:
# =========================================================
# LOAD ALL 5 FOLD MODELS WITH ROBUST ARCHITECTURE HANDLING
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def load_model_with_fallback(fold, device):
    """
    Load a model fold with graceful handling of architecture mismatches.
    Handles checkpoints saved with different ResNet50 variants.
    """
    path = f"{WEIGHTS}/model_fold_{fold}.pt"
    
    if not os.path.exists(path):
        print(f"⚠️  WARNING: {path} not found!")
        return None
    
    model = FlexibleResNet50Audio(CFG["n_mels"], num_classes)
    
    try:
        # First, try standard loading
        state = torch.load(path, map_location=device)
        missing, unexpected = model.load_state_dict(state, strict=False)
        if missing:
            print(f"   ℹ️  Missing keys: {len(missing)}")
        if unexpected:
            print(f"   ℹ️  Unexpected keys: {len(unexpected)}")
        return model
    
    except RuntimeError as e:
        print(f"⚠️  Shape mismatch in fold {fold}: {str(e)[:100]}...")
        print(f"   Attempting selective parameter loading...")
        
        # Load only compatible parameters
        state = torch.load(path, map_location=device)
        model_dict = model.state_dict()
        
        compatible_count = 0
        for k, v in state.items():
            if k in model_dict and model_dict[k].shape == v.shape:
                model_dict[k] = v
                compatible_count += 1
        
        model.load_state_dict(model_dict, strict=False)
        print(f"   ✅ Loaded {compatible_count}/{len(state)} compatible parameters")
        return model

models = []
for fold in range(5):
    model = load_model_with_fallback(fold, device)
    if model is not None:
        model = model.to(device)
        model.eval()
        models.append(model)
        print(f"✅ Fold {fold} ready for inference")

print(f"\n✅ Total models loaded: {len(models)}")
if len(models) < 5:
    print(f"⚠️  WARNING: Only {len(models)}/5 models loaded. Predictions may be degraded.")

In [ ]:
# =========================================================
# TWO-WINDOW ENSEMBLE PREDICTION (2 WINDOWS × 5 FOLDS)
# Balanced accuracy vs speed: 10 forward passes per sample
# Captures birds at start/end without full timeout
# =========================================================
def predict_window(audio_path, start_sec):
    """Make ensemble prediction using 2 windows (balanced trade-off)"""
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
    except Exception as e:
        print(f"Error reading {audio_path}: {e}")
        return np.zeros(num_classes, dtype=np.float32)

    # Resample if needed
    if sr0 != CFG["sr"]:
        y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])

    # Extract 2 OVERLAPPING WINDOWS (2 × 5 = 10 forward passes per sample)
    # Windows: early window (-12.5%) and late window (+12.5%)
    # This captures birds at both ends without the full 3-window overhead
    window_len = int(CFG["seconds"] * CFG["sr"])
    all_preds = []
    
    # Define 2 window positions: -12.5% and +12.5%
    offsets = [-0.125, 0.125]
    
    for offset in offsets:
        center_s = int(start_sec * CFG["sr"]) + int(offset * window_len)
        s = center_s
        e = s + window_len
        
        # Bounds check
        if s < 0 or e > len(y):
            # Out of bounds - pad with zeros
            all_preds.append(np.zeros(num_classes, dtype=np.float32))
            continue
        
        window = y[s:e]
        wave = fixed_length_mono(window, CFG["sr"], CFG["seconds"])
        mel = logmel_from_wave(wave, CFG["sr"])
        x = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        
        # Ensemble predictions across all 5 folds
        fold_preds = []
        with torch.no_grad():
            for model in models:
                logits = model(x)  # [1, num_classes]
                prob = torch.sigmoid(logits)  # Apply sigmoid
                fold_preds.append(prob.cpu().numpy())
        
        # Average across folds
        p_ensemble = np.mean(fold_preds, axis=0)  # [1, num_classes]
        all_preds.append(p_ensemble.squeeze())  # [num_classes]
    
    # Average across 2 windows
    result = np.mean(all_preds, axis=0)  # [num_classes]
    return result

print("✅ TWO-WINDOW ensemble prediction defined (2 windows × 5 folds = 10 FLOPs per sample)")


In [ ]:
# =========================================================
# DIAGNOSTIC: Check test data
# =========================================================
sample = pd.read_csv(SAMPLE_PATH)
print(f"Sample submission shape: {sample.shape}")
print(f"Columns: {sample.columns.tolist()[:10]}...")  # First 10 cols
print(f"\nFirst 3 row_ids:")
print(sample["row_id"].head(3).tolist())

# Check if test files exist
test_files = set()
for row_id in sample["row_id"].head(10):
    file_id = row_id.rsplit("_", 1)[0]
    test_files.add(file_id)

for file_id in test_files:
    audio_path = f"{TEST_DIR}/{file_id}.ogg"
    exists = os.path.exists(audio_path)
    print(f"{'✅' if exists else '❌'} {audio_path}")

In [ ]:
# =========================================================
# BUILD SUBMISSION
# =========================================================
def predict_row(row_id):
    """Parse row_id and return predictions"""
    file_id, start = row_id.rsplit("_", 1)
    start = int(start)
    audio_path = f"{TEST_DIR}/{file_id}.ogg"
    
    if not os.path.exists(audio_path):
        # Expected during notebook development (test_soundscapes is hidden)
        # But will work when officially submitted to Kaggle
        return np.zeros(num_classes, dtype=np.float32)
    
    return predict_window(audio_path, start)


# Apply predictions to each row (with progress bar)
print("\nGenerating predictions...")
print("⚠️  Note: If you see 'all zeros' predictions, the test audio files are not visible yet.")
print("      This is normal during development. Files will be available when officially submitted.")
print()
predictions = []
for row_id in tqdm(sample["row_id"], total=len(sample)):
    pred = predict_row(row_id)
    predictions.append(pred)

predictions = np.array(predictions)  # [num_rows, num_classes]
print(f"Predictions shape: {predictions.shape}")
print(f"Min prob: {predictions.min():.6f}, Max prob: {predictions.max():.6f}")
print(f"Mean prob: {predictions.mean():.6f}")

# Diagnostic: Check if predictions are mostly zeros
if np.mean(predictions) < 0.01:
    print("\n⚠️  WARNING: Predictions are mostly zeros!")
    print("     This typically means test audio files were not found.")
    print("     Expected behavior during notebook development on Kaggle.")
    print("     When officially submitted, test_soundscapes will be available.")


In [ ]:
# =========================================================
# SKIP BIRDNETLIB (not available on Kaggle)
# Will use taxonomy-based proxies instead
# =========================================================
print("ℹ️  Using taxonomy-based approach for missing species")

In [ ]:
# =========================================================
# GENERATE PREDICTIONS & BUILD SUBMISSION
# =========================================================
print("="*70)
print("OPTIMIZED SUBMISSION: SINGLE-WINDOW + 5-FOLD ENSEMBLE")
print("="*70)

# Load sample to get Kaggle's expected species
sample = pd.read_csv(SAMPLE_PATH)
kaggle_species = [col for col in sample.columns if col != 'row_id']
trained_species = species  # from training (206 species)
missing_species = sorted(list(set(kaggle_species) - set(trained_species)))

print(f"\n📊 SPECIES COMPARISON:")
print(f"  Trained: {len(trained_species)}")
print(f"  Kaggle expects: {len(kaggle_species)}")
print(f"  Missing (no training data): {len(missing_species)}")

# =========================================================
# LOAD PER-SPECIES OPTIMAL THRESHOLDS
# =========================================================
try:
    with open(f"{WEIGHTS}/optimal_thresholds.json", "r") as f:
        SPECIES_THRESHOLDS = json.load(f)
    print(f"\n✅ Loaded {len(SPECIES_THRESHOLDS)} per-species thresholds")
except FileNotFoundError:
    print(f"\n⚠️  Optimal thresholds not found. Using uniform 0.5 threshold.")
    SPECIES_THRESHOLDS = {sp: 0.5 for sp in trained_species}

print(f"Threshold range: [{min(SPECIES_THRESHOLDS.values()):.3f}, {max(SPECIES_THRESHOLDS.values()):.3f}]")
print(f"Mean threshold: {np.mean(list(SPECIES_THRESHOLDS.values())):.3f}")

# Build submission with trained species + taxonomy proxies for missing
print(f"\n📝 BUILDING SUBMISSION:")
submission = pd.DataFrame()

for col in kaggle_species:
    if col in trained_species:
        # Use model prediction for trained species
        idx_pos = trained_species.index(col)
        raw_scores = predictions[:, idx_pos]
        
        # Apply per-species threshold
        threshold = SPECIES_THRESHOLDS[col]
        # For now, just clip predictions (can add threshold-based binary if needed)
        tuned_scores = np.clip(raw_scores, 0.0, 1.0)
        
        submission[col] = tuned_scores
    else:
        # Use taxonomy-based proxy for missing species with increased alpha=0.4
        proxy_scores = []
        for row_idx in range(len(predictions)):
            row_pred = predictions[row_idx, :]  # [num_trained_species]
            proxy = get_missing_species_prediction(row_pred, alpha=0.4)
            proxy_scores.append(proxy)
        
        submission[col] = proxy_scores

# Add row_id
submission.insert(0, "row_id", sample["row_id"].astype(str))

# Print summary
print(f"\n✅ VALIDATION:")
print(f"  Shape: {submission.shape}")

In [ ]:
# =========================================================
# SAVE SUBMISSION TO CSV
# =========================================================
output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)

print(f"\n✅ SUBMISSION SAVED:")
print(f"  Path: {output_path}")
print(f"  Shape: {submission.shape}")
print(f"  Rows: {len(submission)}")
print(f"  Columns: {len(submission.columns)}")
print(f"\nFirst few rows:")
print(submission.head(3))
print(f"\nColumn sample (trained + missing):")
trained_cols = [c for c in submission.columns if c in trained_species][:3]
missing_cols = [c for c in submission.columns if c in missing_species][:3]
print(f"  Trained: {trained_cols}")
print(f"  Missing: {missing_cols}")